<a href="https://colab.research.google.com/github/AlHartMos/IEU_courses/blob/main/principals_of_programming/PP_fundamentals_recursion_generators_class.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Advanced Function Patterns: Recursion & Generators

In this notebook we level up from “functions as tools” to “functions as *control structures*”.

You’ll learn two advanced patterns that are everywhere in Computer Science:
- **Recursion**: functions that call themselves (frames **stack up**)
- **Generators (`yield`)**: functions that *pause and resume* (a frame is **saved**)

We’ll keep the focus on **understanding and evidence**:
- predict → run → explain
- write tests for edge cases
- debug wrong code (including wrong AI code)

**Prereqs**: you already know basic functions, loops, and lists.

---
## How to use this notebook
- Micro-checks: do **AI-free** if possible.
- For coding exercises: AI is allowed, but your grade (and your learning) is in your *reasoning* and *tests*.


## Helper: tiny test runner
We’ll use `check(...)` so tests read nicely.


In [1]:
def check(condition, msg=""):
    if not condition:
        raise AssertionError(msg)
    return "✅"

print(check(2 + 2 == 4, "math broke"))

✅


# 1) Recursion: the idea
A **recursive** function solves a problem by solving *smaller versions* of the same problem.

Every correct recursive function needs:
1. **Base case**: stops the recursion
2. **Progress**: each call moves closer to the base case
3. **Recursive step**: calls itself on a smaller input

If you forget (1) or (2), you get infinite recursion (until Python raises a `RecursionError`).

### Mental model: a stack of frames
Each recursive call creates a new **frame** with its own local variables.
Frames stack up until the base case returns, then they **unwind**.


## Example 1: factorial
Mathematically:
- `0! = 1` (base case)
- `n! = n * (n-1)!` for `n > 0`

Let’s implement it and then trace it.


In [2]:
def factorial(n):
    if n < 0:
        raise ValueError("factorial is undefined for negative integers")
    if n == 0:
        return 1
    return n * factorial(n - 1)

print(factorial(5))  # 120

120


### Visual trace (ENTER/EXIT)
This prints the call stack growing and shrinking.


In [3]:
def factorial_trace(n, depth=0):
    indent = "  " * depth
    print(f"{indent}ENTER factorial({n})")

    if n < 0:
        raise ValueError("factorial is undefined for negative integers")
    if n == 0:
        print(f"{indent}BASE -> return 1")
        return 1

    result = n * factorial_trace(n - 1, depth + 1)
    print(f"{indent}EXIT  factorial({n}) -> {result}")
    return result

print("result =", factorial_trace(4))

ENTER factorial(4)
  ENTER factorial(3)
    ENTER factorial(2)
      ENTER factorial(1)
        ENTER factorial(0)
        BASE -> return 1
      EXIT  factorial(1) -> 1
    EXIT  factorial(2) -> 2
  EXIT  factorial(3) -> 6
EXIT  factorial(4) -> 24
result = 24


### Micro-check 1 (predict → run)
Without running first, predict the output order:

1. Does `print('A')` happen before `print('B')`?
2. How many times does `'tick'` print?


In [4]:
def ping(n):
    if n == 0:
        return
    print("tick", n)
    ping(n - 1)
    print("tock", n)

print("A") # Printed before B
ping(3) # Prints tick tick tick tock tock tock
print("B")

A
tick 3
tick 2
tick 1
tock 1
tock 2
tock 3
B


### Exercise 0 — Implement a function to produce a Fibonacci series

The Fibonacci numbers are defined by:

- `fib(0) = 0`
- `fib(1) = 1`
- `fib(n) = fib(n-1) + fib(n-2)` for `n >= 2`

**Your task**
Write `fib(n)` using the definition above (recursive).  

In [9]:
# Your turn:
def fib(n):
    """Return the nth Fibonacci number using recursion.

    Requirements:
    - n is an int, n >= 0
    - Use recursion
    """
    # TODO
    if n == 0:
      return 0
    if n == 1:
      return 1
    return fib(n - 1) + fib(n - 2)

# Evidence (tests)
assert fib(0) == 0
assert fib(1) == 1
assert fib(6) == 8

# 2) Debugging recursion: the 3 classic bugs
When recursion fails, it’s almost always one of these:

1. **Missing base case** → never stops
2. **No progress** → calls itself with the same (or bigger) input
3. **Wrong base case** → stops too early or returns wrong value

Let’s practice with a buggy function.


### Exercise 1 — Fix the buggy sum
The function below is supposed to compute the sum of a list using recursion.

Tasks:
1) Run it and observe the error.
2) Explain the bug in one sentence.
3) Fix it.

Hint: what should the base case be when the list is empty?


In [12]:
def sum_recursive_bug(nums):
    # BUGGY on purpose
    if len(nums) == 0:
      return 0
    if len(nums) == 1:
        return nums[0]
    return nums[0] + sum_recursive_bug(nums[1:])

print(sum_recursive_bug([10]))

10


### Exercise 2 — Recursive palindrome (strings)
Write `is_palindrome_recursive(s)` that returns `True` if `s` reads the same forward and backward.

Requirements:
- Treat the string as-is (no lowercasing, no removing spaces here).
- Use recursion.

Idea:
- Base case: length 0 or 1
- Compare first and last characters, then recurse on the middle.


In [14]:
def is_palindrome_recursive(s):
    # TODO
    if len(s) == 0:
      return True
    if len(s) == 1:
      return True
    return (s[0] == s[len(s)-1]) and is_palindrome_recursive(s[1:(len(s)-2)])
    pass

check(is_palindrome_recursive("") is True)
check(is_palindrome_recursive("a") is True)
check(is_palindrome_recursive("abba") is True)
check(is_palindrome_recursive("abc") is False)
print("palindrome tests passed")

palindrome tests passed


# 3) Recursion on *structures*: nested lists
Recursion shines when the data itself is recursive (a structure containing smaller versions of itself).

A **nested list** is a classic example: items can be numbers or other lists.

Example:
```python
data = [1, [2, 3], [4, [5]]]
```

This looks like a tree.


### Example: count how many integers are inside a nested list
We traverse the structure and sum counts.


In [15]:
def count_ints(nested):
    total = 0
    for item in nested:
        if isinstance(item, int):
            total += 1
        else:
            total += count_ints(item)
    return total

data = [1, [2, 3], [4, [5, 6], []]]
print(count_ints(data))  # 6

6


### Exercise 3 — Flatten a nested list
Write `flatten(nested)` that returns a *flat* list of all integers in left-to-right order.

Example:
- `flatten([1, [2, 3], [4, [5]]]) -> [1, 2, 3, 4, 5]`

Use recursion.


In [23]:
def flatten(nested):
    result = []

    for item in nested:
        if isinstance(item, list):
            result.extend(flatten(item))
        else:
            result.append(item)

    return result


check(flatten([]) == [])
check(flatten([1, [2, 3], [4, [5]]]) == [1, 2, 3, 4, 5])
check(flatten([[[]], 7, [8, [9]]]) == [7, 8, 9])
print("flatten tests passed")

flatten tests passed


# 5) Iteration model: iterables, iterators, `next()`
Before generators, we need one concept:

- An **iterable** is something you can loop over (like a list).
- An **iterator** is an object that produces values one at a time via `next()`.

When you do:
```python
for x in items:
    ...
```
Python roughly does:
```python
it = iter(items)
while True:
    x = next(it)  # until StopIteration
```

Generators are the easiest way to create iterators.


### Micro-check 4 (predict → run)
What will the three `next(...)` calls return?
What happens after the iterator is exhausted?


In [ ]:
it = iter([10, 20, 30])
print(next(it))
print(next(it))
print(next(it))

try:
    print(next(it))
except Exception as e:
    print(type(e).__name__, "->", e)

# 6) Generators: `yield` (functions that pause)
A **generator function** looks like a normal function, but it uses `yield`.

Key idea:
- `return` ends the function.
- `yield` produces a value **and pauses** the function.
- Next time you ask for a value (`next()` or `for`), it resumes exactly where it paused.

So recursion is about frames **stacking**; generators are about a frame being **suspended**.


## Example: countdown generator
Notice: nothing runs until you iterate.


In [ ]:
def countdown(n):
    while n > 0:
        yield n
        n -= 1
    yield "🚀 blastoff!"

g = countdown(3)
print("g is:", g)              # generator object
print(next(g))                 # start running until first yield
print(next(g))
print(next(g))
print(next(g))

## Example: consume with a for-loop
A for-loop repeatedly calls `next()` until `StopIteration`.


In [ ]:
for x in countdown(5):
    print(x)

### Exercise 5 — `range_like` generator
Write `range_like(start, stop, step)` that yields numbers like `range`.

Requirements:
- Works for positive step and negative step
- Raises `ValueError` if `step == 0`

Examples:
- `list(range_like(0, 5, 2)) -> [0, 2, 4]`
- `list(range_like(5, 0, -2)) -> [5, 3, 1]`


In [ ]:
def range_like(start, stop, step):
    # TODO
    pass

check(list(range_like(0, 5, 2)) == [0, 2, 4])
check(list(range_like(5, 0, -2)) == [5, 3, 1])
print("range_like tests passed")

# 7) Generator pipelines (map/filter) + `yield from`
Generators are great for building **pipelines**:
- produce values lazily
- avoid building big intermediate lists

We’ll implement tiny versions of `map` and `filter` using `yield`.


In [ ]:
def gen_map(f, it):
    for x in it:
        yield f(x)

def gen_filter(pred, it):
    for x in it:
        if pred(x):
            yield x

nums = range(10)
pipeline = gen_map(lambda x: x*x, gen_filter(lambda x: x % 2 == 0, nums))
print(list(pipeline))  # squares of even numbers

## `yield from`: delegate to another generator
`yield from subgen` means: yield everything produced by `subgen`.

It’s especially elegant for recursive generators.


### Example: flatten as a *generator* (ties recursion + generators)
Instead of returning a big list, we yield values one by one.


In [ ]:
def flatten_gen(nested):
    for item in nested:
        if isinstance(item, int):
            yield item
        else:
            # delegate to another generator
            yield from flatten_gen(item)

data = [1, [2, 3], [4, [5]]]
print(list(flatten_gen(data)))

### Exercise 6 — unique_consecutive
Write a generator `unique_consecutive(it)` that removes **consecutive duplicates**.

Example:
- input:  `['a','a','b','b','b','a']`
- output: `['a','b','a']`

This is like compressing runs.


In [ ]:
def unique_consecutive(it):
    # TODO
    pass

check(list(unique_consecutive(['a','a','b','b','b','a'])) == ['a','b','a'])
check(list(unique_consecutive([])) == [])
check(list(unique_consecutive([1,1,1])) == [1])
print("unique_consecutive tests passed")

# 8) Recursion vs generators: what’s the *real* difference?
Both use **frames**, but differently:

- **Recursion**: many frames exist at once (stack grows).
- **Generator**: one frame is paused and resumed (state is saved).

A recursive function’s *output* typically appears only when it finishes.
A generator produces output **incrementally**.

If you’re processing huge data, generators can be a big win.

If your data is naturally tree-like, recursion can be the clearest logic.


# 9) Advanced exercises (fun + hard)
These are meant to be challenging. Use:
- a clear contract
- tests
- small examples

### Advanced A — Find a Path Between Two Nodes in a Graph (Lists Only)

In this notebook you’ll practice **graph traversal** and **path reconstruction** using only Python fundamentals (lists, loops, `if`, functions).

We represent a graph using an **adjacency list**:

- Nodes are integers `0, 1, ..., N-1`
- `graph[u]` is a list of neighbors reachable from node `u`
- The example graph below is **undirected** (if `v` is in `graph[u]`, then `u` is in `graph[v]`)


#### 1) Example graph (N nodes)

Try to read it as “rooms connected by doors”. If `graph[2] = [0, 5]` you can go from room `2` to rooms `0` and `5`.



In [ ]:
# N = 9 nodes: 0..8
graph = [
    [1, 2],       # 0
    [0, 3, 4],    # 1
    [0, 5],       # 2
    [1, 6],       # 3
    [1, 6],       # 4
    [2, 7],       # 5
    [3, 4, 8],    # 6
    [5, 8],       # 7
    [6, 7]        # 8
]



import math
import matplotlib.pyplot as plt

def plot_graph(graph):
    n = len(graph)

    # Place nodes on a circle
    pos = []
    for i in range(n):
        angle = 2 * math.pi * i / n
        pos.append((math.cos(angle), math.sin(angle)))

    fig, ax = plt.subplots()

    # Draw edges (avoid drawing each undirected edge twice)
    for u in range(n):
        for v in graph[u]:
            if u < v:
                x1, y1 = pos[u]
                x2, y2 = pos[v]
                ax.plot([x1, x2], [y1, y2])

    # Draw nodes + labels
    for u in range(n):
        x, y = pos[u]
        ax.scatter(x, y)
        ax.text(x, y, f"  {u}", va="center")

    ax.set_aspect("equal")
    ax.axis("off")
    plt.show()

plot_graph(graph)


#### 3) Exercise — `find_path(graph, start, goal)`

Write a function that returns **one valid path** from `start` to `goal` as a list of nodes, or `None` if no path exists.

**Constraints**
- Use lists + basic control flow.
- Use a list as a **stack** (DFS) and a `parent` list to reconstruct the path.

**Hint:** parent pointers
When you discover a node `v` from `u`, store `parent[v] = u`.  
Once you reach `goal`, walk backward: `goal → parent[goal] → ... → start`, then reverse.

### Advanced B — Recursive binary search
Write `binary_search(sorted_nums, target)` that returns the index of `target`, or `-1` if absent.

Requirements:
- Use recursion.
- No loops.
- Input list is sorted ascending.

Hint: write a helper that works with `lo` and `hi` indices.


In [ ]:
def binary_search(sorted_nums, target):
    # TODO
    pass

check(binary_search([1, 3, 5, 7, 9], 7) == 3)
check(binary_search([1, 3, 5, 7, 9], 2) == -1)
check(binary_search([], 10) == -1)
print("binary_search tests passed")

### Advanced C — A lazy prime generator
Write `primes()` that yields prime numbers forever: 2, 3, 5, 7, 11, ...

Constraints (keep it beginner-friendly):
- Use a simple primality test (trial division).
- It’s OK if it’s not super fast.

Then take the first 10 primes.


In [ ]:
def primes():
    # TODO
    pass

# Take first 10 primes
it = primes()
first10 = [next(it) for _ in range(10)]
print(first10)
check(first10 == [2,3,5,7,11,13,17,19,23,29])
print("prime generator tests passed")

### Advanced C — Compose generators into a pipeline
Build a pipeline that:
1) generates numbers from 1 upward
2) filters to keep only multiples of 3
3) maps them to squares
4) removes consecutive duplicates (shouldn’t happen here, but pretend it could)

Return the first 8 results as a list.

Tip: reuse `gen_filter`, `gen_map`, and `unique_consecutive`.


In [ ]:
def naturals(start=1):
    # TODO
    pass

def pipeline_first8():
    # TODO
    pass

print(pipeline_first8())

---
# Summary
You should now be able to:
- explain recursion using base case + progress
- read recursion traces as a call stack of frames
- debug common recursion bugs
- explain iterables/iterators/StopIteration
- write generator functions with `yield`
- build lazy pipelines with generator combinators
- use `yield from` for elegant recursive generators
